# 模組 2 · 資料處理
## 輸入格式、多資料集、波長與合成資料

學會用各種方式餵資料、平行比較多個資料集、處理波長，以及產生合成光譜。

**對應官方範例**：`examples/user/02_data_handling/`（U01–U06）

## 🚀 環境設定（請先執行下面這格）

本課程使用 **nirs4all 官方範例資料集（sample_data）**。下面這格會：
1. 從 GitHub 下載 nirs4all（內含 0.9 版函式庫與官方光譜資料集）
2. 安裝函式庫
3. 切換到 `examples/` 目錄（裡面有 `sample_data/`）

> 💡 **為什麼從 GitHub 安裝？** PyPI（`pip install nirs4all`）目前是舊版 0.4，沒有本課程使用的 `nirs4all.run()` 簡易 API。GitHub 上的版本（0.9+）才有，且需要 **Python ≥ 3.11**（Colab 預設 3.12，沒問題）。

第一次執行約需 1–2 分鐘，請耐心等候出現「✅ 完成」。

In [ ]:
# === Colab / Jupyter 環境設定（每本 notebook 開頭執行一次）===
import os, sys, subprocess

if not os.path.isdir('nirs4all'):
    print('⬇️  下載 nirs4all 與官方資料集（約 1–2 分鐘）…')
    subprocess.run(['git', 'clone', '-q',
                    'https://github.com/GBeurier/nirs4all.git'], check=True)
    print('📦 安裝 nirs4all …')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                    './nirs4all'], check=True)

# 切換到 examples 目錄（內含 sample_data）
if os.path.basename(os.getcwd()) != 'examples' and os.path.isdir('nirs4all/examples'):
    os.chdir('nirs4all/examples')

import nirs4all
print('✅ 完成！nirs4all 版本：', nirs4all.__version__)
print('   工作目錄：', os.getcwd())
print('   可用資料集：', [d for d in os.listdir('sample_data') if os.path.isdir(os.path.join('sample_data', d))])

---
## U01 · 彈性輸入：各種資料格式  ★☆☆☆☆

`dataset` 參數很彈性：可給資料夾路徑、`(X, y)` tuple、字典或 SpectroDataset。`partition_info` 控制切分，例如 `{"train": 160}` 表前 160 筆為訓練。

In [ ]:
import numpy as np, nirs4all
from sklearn.cross_decomposition import PLSRegression

# 用 numpy 陣列快速實驗
X = np.random.rand(200, 100)   # 200 樣本 × 100 波長
y = np.random.rand(200)

result = nirs4all.run(
    pipeline=[PLSRegression(n_components=5)],
    dataset=(X, y, {"train": 160}),   # 前 160 訓練、其餘測試
    name="ArrayInput", verbose=1,
)
print("RMSE:", round(result.best_rmse, 4))

> ✍️ **練習**：把 `{"train": 160}` 改成 `{"train": slice(0,150)}`，確認切分正確。

---
## U02 · 多資料集：平行分析與比較  ★★☆☆☆

把 `dataset` 傳成**清單**，同一條 pipeline 公平套用到每個資料集，方便比較泛化能力。用 `result.per_dataset` 取得各資料集成績。

In [ ]:
import nirs4all
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import ShuffleSplit
from sklearn.cross_decomposition import PLSRegression
from sklearn.linear_model import ElasticNet
from nirs4all.operators.transforms import (
    MultiplicativeScatterCorrection, Gaussian, StandardNormalVariate, SavitzkyGolay, Haar)

pipeline = [
    MinMaxScaler(feature_range=(0.1, 0.8)),
    {"feature_augmentation": [MultiplicativeScatterCorrection, Gaussian,
                              StandardNormalVariate, SavitzkyGolay, Haar]},
    ShuffleSplit(n_splits=3, random_state=42),
    {"y_processing": MinMaxScaler()},
    {"model": PLSRegression(n_components=5),  "name": "PLS-5"},
    {"model": PLSRegression(n_components=10), "name": "PLS-10"},
    {"model": ElasticNet(alpha=0.1),          "name": "ElasticNet"},
]
result = nirs4all.run(
    pipeline=pipeline,
    dataset=['sample_data/regression', 'sample_data/regression_2'],
    name="MultiDataset", verbose=1)
print("\n各資料集：", list(result.per_dataset.keys()))

> ✍️ **練習**：再加 `'sample_data/regression_3'` 到清單，找出三個資料集都穩定的模型。

---
## U03 · 多來源資料 & generator 進階語法  ★★★☆☆

當樣本有多種量測（NIR + 標記、多台儀器）時，nirs4all 自動載入並串接所有來源。本課同時示範進階 generator：`pick`（挑幾種串接）、`then_pick`（串接後再挑變體）、`count`（產生幾組）。

In [ ]:
import nirs4all
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import ShuffleSplit
from sklearn.cross_decomposition import PLSRegression
from nirs4all.operators.transforms import (
    StandardNormalVariate, SavitzkyGolay, Gaussian, Haar)

pipeline = [
    MinMaxScaler(),
    {"y_processing": MinMaxScaler()},
    {"feature_augmentation": {
        "_or_": [StandardNormalVariate(), SavitzkyGolay(), Gaussian(), Haar()],
        "pick": [2, 3],        # 挑 2 或 3 種前處理
        "then_pick": [1, 3],   # 串接後再挑 1~3 種變體
        "count": 2,            # 產生 2 組
    }},
    ShuffleSplit(n_splits=3, random_state=42),
    MinMaxScaler(feature_range=(0.1, 0.8)),
    {"model": PLSRegression(n_components=10), "name": "PLS-10"},
]
result = nirs4all.run(pipeline=pipeline, dataset="sample_data/regression",
                      name="MultiSource", verbose=1)
print("產生變體數：", result.num_predictions)

> ✍️ **練習**：把 `pick` 改成固定 `2`，觀察變體數如何變化。

---
## U04 · 波長處理：重取樣與單位  ★★☆☆☆

`Resampler` 把光譜重新取樣到指定波長格點：可用於儀器標準化、降維、區段聚焦。內插方法：linear / cubic / quadratic / nearest。

In [ ]:
import numpy as np, nirs4all
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import ShuffleSplit
from sklearn.cross_decomposition import PLSRegression
from nirs4all.operators.transforms import Resampler

# 把光譜降採樣到 60 個點後建模
target_wl = np.linspace(1000, 2500, 60)

pipeline = [
    Resampler(target_wavelengths=target_wl, method='cubic'),
    MinMaxScaler(),
    {"y_processing": MinMaxScaler()},
    ShuffleSplit(n_splits=3, test_size=0.25, random_state=42),
    {"model": PLSRegression(n_components=10)},
]
result = nirs4all.run(pipeline=pipeline, dataset="sample_data/regression",
                      name="Resample", verbose=1)
print("降採樣後 RMSE:", round(result.best_rmse, 4))

> ✍️ **練習**：把 60 改成 30 點再跑，比較與全波長的 RMSE，評估降維代價。

---
## U05 · 合成資料：generate()  ★★☆☆☆

還沒有真實資料？用 `nirs4all.generate()` 產生具已知真值的合成光譜，適合測試與教學。關鍵參數：`complexity`（simple/realistic/complex）、`components`（water、protein…）。

In [ ]:
import nirs4all
from sklearn.cross_decomposition import PLSRegression
from sklearn.model_selection import ShuffleSplit

# 產生 (X, y) 合成回歸資料
X, y = nirs4all.generate.regression(
    n_samples=300, complexity="realistic",
    components=["water", "protein"], random_state=42, as_dataset=False)
print("合成資料形狀:", X.shape, y.shape)

result = nirs4all.run(
    pipeline=[ShuffleSplit(n_splits=3, test_size=0.25),
              {"model": PLSRegression(n_components=10)}],
    dataset=(X, y), name="Synthetic", verbose=1)
print("RMSE:", round(result.best_rmse, 4))

> ✍️ **練習**：分別用 `complexity="simple"` 與 `"complex"` 產生資料訓練，觀察 R² 落差。

---
## U06 · 進階合成資料：Builder API  ★★★☆☆

`SyntheticDatasetBuilder` 提供鏈式 API 精細控制特徵、目標、metadata、批次效應與非線性目標，可打造接近真實的困難基準。

In [ ]:
import nirs4all
from nirs4all.data.synthetic import SyntheticDatasetBuilder
from sklearn.cross_decomposition import PLSRegression
from sklearn.model_selection import ShuffleSplit

ds = (SyntheticDatasetBuilder(n_samples=400, random_state=42, name="demo")
      .with_features(wavelength_range=(1000, 2500), complexity="realistic",
                     components=["water", "protein", "lipid"])
      .with_targets(distribution="normal", range=(0, 100), component="protein")
      .with_metadata(n_groups=8, n_repetitions=2)
      .with_batch_effects(enabled=True, n_batches=4)
      .with_partitions(train_ratio=0.8)
      .build())

result = nirs4all.run(
    pipeline=[ShuffleSplit(n_splits=3, test_size=0.25),
              {"model": PLSRegression(n_components=10)}],
    dataset=ds, name="BuilderDemo", verbose=1)
print("RMSE:", round(result.best_rmse, 4))

> ✍️ **練習**：加上 `.with_nonlinear_targets(interactions="synergistic", hidden_factors=2)`，看非線性目標讓 PLS 的 R² 掉多少。